<a href="https://colab.research.google.com/github/kidujm/laptop-data-analysis/blob/main/%EB%8B%A4%EB%82%98%EC%99%80_%EC%A0%84%EC%B2%98%EB%A6%AC3_0319_%EC%A0%95%EB%AF%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
lap = pd.read_csv('danawa_with_date.csv')
lap.head()
lap.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864 entries, 0 to 863
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   제품명        864 non-null    object 
 1   브랜드        864 non-null    object 
 2   CPU브랜드     844 non-null    object 
 3   CPU등급      831 non-null    object 
 4   그래픽        864 non-null    object 
 5   OS         864 non-null    object 
 6   별점         571 non-null    float64
 7   배터리용량      864 non-null    float64
 8   무게(kg)     864 non-null    float64
 9   가격         864 non-null    int64  
 10  RAM(GB)    864 non-null    float64
 11  AI여부       864 non-null    int64  
 12  화면크기(인치)   864 non-null    float64
 13  용도         864 non-null    object 
 14  리뷰수        864 non-null    int64  
 15  용도_게임용     864 non-null    int64  
 16  용도_그래픽작업용  864 non-null    int64  
 17  용도_사무      864 non-null    int64  
 18  용도_휴대용     864 non-null    int64  
 19  등록일        864 non-null    object 
dtypes: float64

In [ ]:
import pandas as pd

lap = pd.read_csv('danawa_with_date.csv')

print(f"전체: {len(lap)}개")
print(f"등록일 있음: {lap['등록일'].notna().sum()}개")
print(f"등록일 없음 (결측): {lap['등록일'].isna().sum()}개")
print()
print("=== 결측 제품 목록 ===")
print(lap[lap['등록일'].isna()][['제품명', '등록일']])

전체: 864개
등록일 있음: 864개
등록일 없음 (결측): 0개

=== 결측 제품 목록 ===
Empty DataFrame
Columns: [제품명, 등록일]
Index: []


In [ ]:
danawa3 = pd.read_csv('danawa2.csv')

In [ ]:
danawa3['등록일'] = lap['등록일'].values

In [ ]:
danawa3.to_csv('danawa3.csv', index=False, encoding='utf-8-sig')

# 클러스터링 전 전처리
- cpu 브랜드 결측 전처리
- 등록일 파싱
- 챗봇 제안  "26.01." 형태를 클러스터링에 쓰려면 숫자로 변환해야 합니다. 현재 기준 경과 개월수로 변환하는 게 가장 자연스럽습니다. (예: 26.01. → 2개월 전)
-  CPU브랜드·OS 인코딩
- 스케일 정규화

In [ ]:
# ① AMD 계열 — 제품명 패턴으로 등급 채우기
cpu_grade_fix = {
    'G614FR': 7,   # Ryzen 7 7745HX
    'G814FM': 7,   # Ryzen 7 7745HX
    'R9'    : 9,   # MSI Ryzen 9
    '16AFR' : 5,   # 레노버 Ryzen 5
}

for pattern, grade in cpu_grade_fix.items():
    mask = danawa3['CPU등급'].isna() & danawa3['제품명'].str.contains(pattern)
    danawa3.loc[mask, 'CPU등급'] = grade

# ② Snapdragon 계열 4개 — CPU브랜드/등급 별도 처리
# (HP 옴니북X, ASUS 젠북A16, 삼성 갤럭시북4 엣지, ASUS 비보북S)
# Snapdragon X Elite → 7등급, Snapdragon X → 5등급으로 매핑
snap_elite = ['fe0005', 'NT960XMB']   # X Elite
snap_std   = ['SQ012W', 'SQ068W']     # X

danawa3.loc[danawa3['제품명'].str.contains('|'.join(snap_elite)), 'CPU브랜드'] = 'Qualcomm'
danawa3.loc[danawa3['제품명'].str.contains('|'.join(snap_elite)), 'CPU등급']   = 7

danawa3.loc[danawa3['제품명'].str.contains('|'.join(snap_std)), 'CPU브랜드'] = 'Qualcomm'
danawa3.loc[danawa3['제품명'].str.contains('|'.join(snap_std)), 'CPU등급']   = 5

# 확인
print(danawa3[danawa3['CPU등급'].isna() | danawa3['CPU브랜드'].isna()][['제품명','CPU브랜드','CPU등급']])

                              제품명 CPU브랜드 CPU등급
695  ASUS 비보북 S 15 S5507QA-MA068W    NaN   NaN


In [ ]:
danawa3 = danawa3.drop(index=695).reset_index(drop=True)

In [ ]:
# 등록일 "26.01." → 현재 기준 경과 개월수로 변환
from datetime import datetime

def parse_date_to_months(date_str):
    try:
        year, month = date_str.strip('.').split('.')
        year = int('20' + year)
        month = int(month)
        reg_date = datetime(year, month, 1)
        now = datetime.now()
        return (now.year - reg_date.year) * 12 + (now.month - reg_date.month)
    except:
        return None

danawa3['등록_경과월'] = danawa3['등록일'].apply(parse_date_to_months)

# 확인
print(danawa3[['등록일', '등록_경과월']].head(10))

      등록일  등록_경과월
0  26.01.       2
1  25.03.      12
2  25.06.       9
3  25.07.       8
4  25.08.       7
5  25.07.       8
6  26.01.       2
7  25.07.       8
8  25.06.       9
9  25.06.       9


In [ ]:
danawa3.to_csv('danawa3_ver2.csv', index=False, encoding='utf-8-sig')

In [ ]:
def visualize_kmeans_plot_multi(cluster_lists, X_features):
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    n_clusters = len(cluster_lists)
    n_cols = min(n_clusters, 5)
    n_rows = math.ceil(n_clusters / n_cols)

    fig, axs = plt.subplots(figsize=(4 * n_cols, 4 * n_rows), nrows=n_rows, ncols=n_cols)

    axs = np.array(axs).reshape(-1) if n_clusters > 1 else [axs]

    pca = PCA(n_components=2)
    pca_transformed = pca.fit_transform(X_features)
    dataframe = pd.DataFrame(pca_transformed, columns=['PCA1', 'PCA2'])

    for ind, n_cluster in enumerate(cluster_lists):

        clusterer = KMeans(n_clusters=n_cluster, max_iter=500, random_state=0)
        cluster_labels = clusterer.fit_predict(pca_transformed)
        dataframe['cluster'] = cluster_labels

        unique_labels = np.unique(cluster_labels)
        markers = ['o', 's', '^', 'x', '*', 'D', 'v', 'p', 'H', '+']

        for label in unique_labels:
            label_df = dataframe[dataframe['cluster'] == label]
            cluster_legend = f'Cluster {label}' if label != -1 else 'Noise'
            axs[ind].scatter(x=label_df['PCA1'], y=label_df['PCA2'], s=70,
                             edgecolor='k', marker=markers[label % len(markers)], label=cluster_legend)

        axs[ind].set_title(f'Number of Clusters: {n_cluster}')
        axs[ind].legend(loc='upper right')

    for ax in axs[n_clusters:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def visualize_silhouette(cluster_lists, X_features):
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_samples, silhouette_score

    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import numpy as np
    import math

    n_clusters = len(cluster_lists)

    n_cols = min(n_clusters, 5)
    n_rows = math.ceil(n_clusters / n_cols)

    fig, axs = plt.subplots(figsize=(4 * n_cols, 4 * n_rows), nrows=n_rows, ncols=n_cols)

    axs = np.array(axs).reshape(-1) if n_clusters > 1 else [axs]

    for ind, n_cluster in enumerate(cluster_lists):

        clusterer = KMeans(n_clusters=n_cluster, max_iter=500, random_state=0)
        cluster_labels = clusterer.fit_predict(X_features)

        sil_avg = silhouette_score(X_features, cluster_labels)
        sil_values = silhouette_samples(X_features, cluster_labels)

        y_lower = 10
        axs[ind].set_title(f'Number of Clusters: {n_cluster}\nSilhouette Score: {round(sil_avg, 3)}')
        axs[ind].set_xlabel("The silhouette coefficient values")
        axs[ind].set_ylabel("Cluster label")
        axs[ind].set_xlim([-0.1, 1])
        axs[ind].set_ylim([0, len(X_features) + (n_cluster + 1) * 10])
        axs[ind].set_yticks([])
        axs[ind].set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])

        for i in range(n_cluster):
            ith_cluster_sil_values = sil_values[cluster_labels == i]
            ith_cluster_sil_values.sort()

            size_cluster_i = ith_cluster_sil_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_cluster)
            axs[ind].fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil_values,
                                   facecolor=color, edgecolor=color, alpha=0.7)
            axs[ind].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
            y_lower = y_upper + 10

        axs[ind].axvline(x=sil_avg, color="red", linestyle="--")

    for ax in axs[n_clusters:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# 정규화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score

In [ ]:
danawa3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 863 entries, 0 to 862
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   제품명         863 non-null    object 
 1   브랜드         863 non-null    object 
 2   AI여부        863 non-null    int64  
 3   CPU브랜드      863 non-null    object 
 4   CPU등급       863 non-null    object 
 5   그래픽         863 non-null    object 
 6   RAM(GB)     863 non-null    float64
 7   OS          863 non-null    object 
 8   배터리용량       863 non-null    float64
 9   무게(kg)      863 non-null    float64
 10  화면크기(인치)    863 non-null    float64
 11  가격          863 non-null    int64  
 12  리뷰수         863 non-null    int64  
 13  용도          863 non-null    object 
 14  용도_게임용      863 non-null    int64  
 15  용도_그래픽작업용   863 non-null    int64  
 16  용도_사무       863 non-null    int64  
 17  용도_휴대용      863 non-null    int64  
 18  has_rating  863 non-null    int64  
 19  별점          863 non-null    f

In [ ]:
# CPU등급 문자열 → 숫자 매핑
cpu_grade_map = {
    '3.0': 3, '5.0': 5, '7.0': 7, '9.0': 9,
    3: 3, 5: 5, 7: 7, 9: 9,
    'M2': 5, 'M3': 5, 'M3프로': 7,
    'M4': 7, 'M4프로': 7,
    'M5': 7, 'M5프로': 9,
}

danawa3['CPU등급'] = danawa3['CPU등급'].map(cpu_grade_map)

# 확인
print(danawa3['CPU등급'].value_counts(dropna=False))

CPU등급
7.0    399
5.0    237
9.0    209
NaN     17
3.0      2
Name: count, dtype: int64


In [ ]:
# 수치형으로 변환해야 할 컬럼들 상태 확인
num_cols = ['RAM(GB)', '배터리용량', '무게(kg)', '화면크기(인치)', '가격', '리뷰수', '별점', 'CPU등급', '등록_경과월']

for col in num_cols:
    print(f"[{col}]")
    print(f"  dtype: {danawa3[col].dtype}")
    print(f"  결측: {danawa3[col].isna().sum()}개")
    print(f"  고유값 샘플: {danawa3[col].unique()[:5]}")
    print()

[RAM(GB)]
  dtype: float64
  결측: 0개
  고유값 샘플: [ 32.  16.  24. 128.   8.]

[배터리용량]
  dtype: float64
  결측: 0개
  고유값 샘플: [77. 83. 90. 80. 73.]

[무게(kg)]
  dtype: float64
  결측: 0개
  고유값 샘플: [1.19  1.199 2.35  2.45  1.9  ]

[화면크기(인치)]
  dtype: float64
  결측: 0개
  고유값 샘플: [16.  15.1 14.  17.3 15.6]

[가격]
  dtype: int64
  결측: 0개
  고유값 샘플: [3199000 2489990 2399000 2893000 1969000]

[리뷰수]
  dtype: int64
  결측: 0개
  고유값 샘플: [102 148  31   7  43]

[별점]
  dtype: float64
  결측: 0개
  고유값 샘플: [4.9 4.8 5.  4.6 4.4]

[CPU등급]
  dtype: object
  결측: 0개
  고유값 샘플: ['7.0' '9.0' '5.0' 'M5프로' 'M5']

[등록_경과월]
  dtype: int64
  결측: 0개
  고유값 샘플: [ 2 12  9  8  7]



In [ ]:
num_cols = ['RAM(GB)', '배터리용량', '무게(kg)', '화면크기(인치)', '가격', '리뷰수', '별점', 'CPU등급', '등록_경과월']

scaler = StandardScaler()
danawa3_scaled = danawa3.copy()
danawa3_scaled[num_cols] = scaler.fit_transform(danawa3[num_cols])

print(danawa3_scaled[num_cols].describe().round(2))

       RAM(GB)   배터리용량  무게(kg)  화면크기(인치)      가격     리뷰수      별점   CPU등급  \
count   863.00  863.00  863.00    863.00  863.00  863.00  863.00  863.00   
mean      0.00    0.00    0.00      0.00   -0.00   -0.00   -0.00    0.00   
std       1.00    1.00    1.00      1.00    1.00    1.00    1.00    1.00   
min      -1.31   -2.16   -1.70     -2.57   -1.76   -0.20   -1.37   -2.69   
25%      -0.79   -1.11   -0.81     -0.32   -0.67   -0.20   -1.37   -1.32   
50%       0.25    0.06   -0.11      0.08   -0.17   -0.18    0.69    0.05   
75%       0.25    0.88    0.82      0.08    0.44   -0.13    0.82    0.05   
max       6.47    1.76    4.05      2.43    4.63   13.08    0.82    1.43   

       등록_경과월  
count  863.00  
mean     0.00  
std      1.00  
min     -1.29  
25%     -0.70  
50%     -0.33  
75%      0.63  
max      3.29  


# 클러스터링하기전 정규화

In [ ]:
danawa3_scaled.to_csv('danawa3_scaled.csv', index=False, encoding='utf-8-sig')